# De Regresión Logística al Perceptrón Multicapa
### Clase Previa al Notebook de Redes y Transformers
**Facultad de Ciencias, UNAM — Proyecto I: Introducción a LLMs — Semestre 2026-2**

---

> **Objetivo:** Demostrar que una neurona artificial *es* regresión logística, identificar cuándo falla, y motivar las redes profundas que están dentro de todo LLM.

**Mapa de la sesión (≈ 60 min):**

```
Parte 1: Regresión Logística como una neurona         (~20 min)
Parte 2: ¿Cuándo falla? El problema de la linealidad  (~15 min)
Parte 3: La solución — agregar capas ocultas          (~15 min)
Parte 4: Conexión directa con el Transformer          (~10 min)
```

**Todo corre en CPU.** No se necesita GPU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

torch.manual_seed(0)
np.random.seed(0)

print("Listo. PyTorch:", torch.__version__)

---
## Parte 1: Regresión Logística — Lo que ya conocen

### 1.1 El modelo

La regresión logística modela la probabilidad de que una observación $\mathbf{x} \in \mathbb{R}^d$ pertenezca a la clase 1:

$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^\top \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^\top \mathbf{x} + b)}}$$

Los **parámetros** son $\mathbf{w} \in \mathbb{R}^d$ y $b \in \mathbb{R}$.

La **frontera de decisión** es el hiperplano $\{\mathbf{x} : \mathbf{w}^\top \mathbf{x} + b = 0\}$, donde $\sigma = 0.5$.

### 1.2 El mismo modelo como neurona

```
x₁ ──(w₁)──┐
x₂ ──(w₂)──┤──► z = w·x + b ──► σ(z) = ŷ
x₃ ──(w₃)──┘
```

**Son exactamente la misma cosa.** Una neurona con activación sigmoide *es* regresión logística binaria.

In [ ]:
# ── Implementación desde cero (solo numpy) ──────────────────────────────────
# Tarea: clasificar reseñas de texto como positivas / negativas
# Features: [prop_palabras_positivas, prop_palabras_negativas]

# Dataset sintético — separable linealmente
np.random.seed(1)
n = 80

# Clase 0: reseñas negativas
X0 = np.random.randn(n//2, 2) + np.array([-1.5, 1.5])
# Clase 1: reseñas positivas
X1 = np.random.randn(n//2, 2) + np.array([1.5, -1.5])

X = np.vstack([X0, X1]).astype(np.float32)
y = np.array([0]*( n//2) + [1]*(n//2), dtype=np.float32)

# Regresión logística implementada manualmente
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def cross_entropy(y_true, y_hat):
    """Binary Cross-Entropy: -[y·log(ŷ) + (1-y)·log(1-ŷ)]"""
    eps = 1e-8  # evitar log(0)
    return -np.mean(y_true * np.log(y_hat + eps) + (1 - y_true) * np.log(1 - y_hat + eps))

def gradiente(X, y, w, b):
    """Gradiente de la cross-entropy respecto a w y b."""
    m = len(y)
    y_hat = sigmoid(X @ w + b)
    error = y_hat - y
    dw = (X.T @ error) / m      # ∂L/∂w
    db = np.mean(error)          # ∂L/∂b
    return dw, db

# Inicializar parámetros
w = np.zeros(2)
b = 0.0
lr = 0.3
historial = []

# Descenso de gradiente (200 pasos)
for _ in range(200):
    y_hat = sigmoid(X @ w + b)
    loss = cross_entropy(y, y_hat)
    historial.append(loss)
    dw, db = gradiente(X, y, w, b)
    w -= lr * dw
    b -= lr * db

y_pred = (sigmoid(X @ w + b) >= 0.5).astype(int)
acc = np.mean(y_pred == y)
print(f"Loss final: {historial[-1]:.4f}")
print(f"Accuracy:   {acc*100:.1f}%")
print(f"Parámetros aprendidos: w = {w.round(3)}, b = {b:.3f}")

In [ ]:
# Visualización: frontera de decisión
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Curva de aprendizaje ──
axes[0].plot(historial, color='#E53935', linewidth=2)
axes[0].set_title('Curva de aprendizaje', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Paso de gradiente')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)

# ── Datos + frontera de decisión ──
x1_range = np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 200)
x2_range = np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 200)
xx1, xx2 = np.meshgrid(x1_range, x2_range)
Z = sigmoid(np.c_[xx1.ravel(), xx2.ravel()] @ w + b).reshape(xx1.shape)

axes[1].contourf(xx1, xx2, Z, levels=50, cmap='RdYlBu_r', alpha=0.4)
axes[1].contour(xx1, xx2, Z, levels=[0.5], colors='black', linewidths=2,
                linestyles='--', label='Frontera (σ=0.5)')
axes[1].scatter(X0[:, 0], X0[:, 1], c='#1565C0', s=40, label='Negativa (0)', zorder=3)
axes[1].scatter(X1[:, 0], X1[:, 1], c='#C62828', s=40, label='Positiva (1)', zorder=3)
axes[1].set_title('Frontera de decisión\n(Regresión Logística = 1 neurona)', 
                   fontsize=11, fontweight='bold')
axes[1].set_xlabel('prop. palabras positivas')
axes[1].set_ylabel('prop. palabras negativas')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print("La frontera es un HIPERPLANO (línea en 2D): w·x + b = 0")
print("Esta es la limitación fundamental que motivará las redes profundas.")

### 1.3 ¿Por qué Cross-Entropy? Derivación desde MLE

Asumimos que $y_i \mid \mathbf{x}_i \sim \text{Bernoulli}(\hat{y}_i)$. La log-verosimilitud es:

$$\log \mathcal{L}(\mathbf{w}, b) = \sum_{i=1}^m \left[ y_i \log \hat{y}_i + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

**Maximizar** la log-verosimilitud es equivalente a **minimizar** la cross-entropy negada:

$$\mathcal{L}_{CE} = -\frac{1}{m} \sum_{i=1}^m \left[ y_i \log \hat{y}_i + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

El gradiente tiene una forma sorprendentemente limpia:

$$\frac{\partial \mathcal{L}_{CE}}{\partial \mathbf{w}} = \frac{1}{m} X^\top (\hat{\mathbf{y}} - \mathbf{y})$$

El error simplemente se propaga hacia atrás — esta es la intuición de backpropagation.

In [ ]:
# ── Lo mismo en PyTorch ──────────────────────────────────────────────────────
# nn.Linear + sigmoid + BCELoss = regresión logística

X_t = torch.tensor(X)
y_t = torch.tensor(y)

class RegresionLogistica(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.lineal = nn.Linear(d, 1)  # w·x + b
    
    def forward(self, x):
        return torch.sigmoid(self.lineal(x)).squeeze()

modelo_rl = RegresionLogistica(d=2)
optimizador = torch.optim.SGD(modelo_rl.parameters(), lr=0.3)
criterio = nn.BCELoss()

for epoch in range(200):
    y_hat = modelo_rl(X_t)
    loss = criterio(y_hat, y_t)
    optimizador.zero_grad()
    loss.backward()
    optimizador.step()

with torch.no_grad():
    preds = (modelo_rl(X_t) >= 0.5).float()
    acc_pt = (preds == y_t).float().mean().item()

print("Regresión Logística en PyTorch:")
print(f"  Loss final: {loss.item():.4f}")
print(f"  Accuracy:   {acc_pt*100:.1f}%")

# Mostrar los parámetros aprendidos
w_pt = modelo_rl.lineal.weight.data[0]
b_pt = modelo_rl.lineal.bias.data[0]
print(f"  w = {w_pt.numpy().round(3)}, b = {b_pt.item():.3f}")
print()
print("nn.Linear(2, 1) tiene exactamente 3 parámetros: w[0], w[1], b")
print("Es la neurona más simple posible.")

---
## Parte 2: ¿Cuándo Falla la Regresión Logística?

### El problema fundamental: linealidad

La regresión logística solo puede crear **fronteras de decisión lineales**. Si los datos no son linealmente separables, el modelo falla.

### El ejemplo clásico: XOR

La tabla XOR no es linealmente separable — ninguna línea recta puede separar correctamente los cuatro puntos:

| $x_1$ | $x_2$ | $x_1 \oplus x_2$ |
|:-----:|:-----:|:-----------------:|
|   0   |   0   |         0         |
|   0   |   1   |         1         |
|   1   |   0   |         1         |
|   1   |   1   |         0         |

Este es el ejemplo que Minsky y Papert usaron en 1969 para "probar" que las redes neuronales eran inútiles. Tenían razón para una sola capa. Una capa oculta lo resuelve.

In [ ]:
# XOR: donde la regresión logística falla

X_xor = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_xor = torch.tensor([0., 1., 1., 0.])

# Entrenar regresión logística en XOR
modelo_xor_rl = RegresionLogistica(d=2)
opt_xor = torch.optim.SGD(modelo_xor_rl.parameters(), lr=0.5)
crit = nn.BCELoss()

losses_rl_xor = []
for _ in range(2000):
    out = modelo_xor_rl(X_xor)
    loss = crit(out, y_xor)
    opt_xor.zero_grad()
    loss.backward()
    opt_xor.step()
    losses_rl_xor.append(loss.item())

with torch.no_grad():
    probs_rl = modelo_xor_rl(X_xor)
    preds_rl = (probs_rl >= 0.5).float()

print("Regresión Logística en XOR (2000 pasos):")
print(f"  Loss final: {losses_rl_xor[-1]:.4f}")
print()
print(f"  {'Punto':>10} | {'Verdad':>8} | {'Predicción':>10} | {'Prob':>8}")
print("  " + "-"*45)
nombres = ["(0,0)", "(0,1)", "(1,0)", "(1,1)"]
for i, (nombre, yv, yp, prob) in enumerate(zip(nombres, y_xor, preds_rl, probs_rl)):
    ok = "✓" if yv == yp else "✗"
    print(f"  {nombre:>10} | {int(yv.item()):>8} | {int(yp.item()):>10} {ok}  | {prob.item():>7.3f}")

In [ ]:
# Visualizar por qué XOR es imposible para un modelo lineal
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colores = ['#1565C0', '#C62828', '#C62828', '#1565C0']
labels_xor = ['(0,0)→0', '(0,1)→1', '(1,0)→1', '(1,1)→0']

# ── Datos XOR ──
for i, (punto, color, label) in enumerate(zip(X_xor, colores, labels_xor)):
    axes[0].scatter(punto[0].item(), punto[1].item(), c=color, s=250, 
                    zorder=5, edgecolors='black', linewidth=1.5)
    axes[0].annotate(label, (punto[0].item(), punto[1].item()),
                     textcoords="offset points", xytext=(8, 8), fontsize=10)

axes[0].set_xlim(-0.5, 1.5)
axes[0].set_ylim(-0.5, 1.5)
axes[0].set_title('Problema XOR\n¿Existe una línea que separe azul de rojo?',
                   fontsize=11, fontweight='bold')
axes[0].set_xlabel('$x_1$', fontsize=12)
axes[0].set_ylabel('$x_2$', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Frontera de decisión aprendida (que no sirve)
xx = np.linspace(-0.5, 1.5, 200)
w_xor = modelo_xor_rl.lineal.weight.data[0]
b_xor = modelo_xor_rl.lineal.bias.data[0]
if abs(w_xor[1].item()) > 1e-4:
    yy = -(w_xor[0].item() * xx + b_xor.item()) / w_xor[1].item()
    axes[0].plot(xx, yy, 'k--', linewidth=2, label='Frontera aprendida', alpha=0.7)
    axes[0].legend(fontsize=9)

# ── Pérdida atascada ──
axes[1].plot(losses_rl_xor, color='#E53935', linewidth=2)
axes[1].axhline(np.log(2), color='gray', linestyle=':', alpha=0.7,
                label=f'ln(2) = {np.log(2):.3f}\n(loss de adivinar al azar)')
axes[1].set_title('La loss se atasca\n(no converge a cero)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Paso')
axes[1].set_ylabel('BCE Loss')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("La regresión logística no puede aprender XOR.")
print("La loss no llega a cero porque ninguna línea recta puede clasificar los 4 puntos.")

### ¿Por qué importa esto para NLP?

En lenguaje natural, la mayoría de los problemas son **no lineales**:

- *"el banco del río"* vs *"el banco financiero"* — la misma palabra requiere contexto
- *"no es malo"* (doble negación → positivo) — composición no lineal
- *"¿puedes cerrar la puerta?"* (solicitud cortés, no pregunta de capacidad)

La regresión logística sobre bag-of-words puede detectar palabras positivas/negativas simples, pero no puede capturar estas interacciones. Necesitamos **funciones no lineales compuestas** — eso es exactamente lo que hacen las redes profundas.

---
## Parte 3: La Solución — Una Capa Oculta

### Idea clave

Una capa oculta aprende una **transformación del espacio de entrada** tal que los datos volviéndose linealmente separables en el nuevo espacio.

Para XOR, la red aprende a **reinterpretar** el espacio $(x_1, x_2)$ en un espacio $(h_1, h_2)$ donde sí existe una frontera lineal.

Matemáticamente, la red computa:

$$\mathbf{h} = \sigma(W_1 \mathbf{x} + \mathbf{b}_1) \in \mathbb{R}^{d_h}$$
$$\hat{y} = \sigma(\mathbf{w}_2^\top \mathbf{h} + b_2) \in \mathbb{R}$$

La capa oculta construye **representaciones intermedias** — *features* que no estaban en los datos originales.

In [ ]:
# Red con una capa oculta — resuelve XOR

class MLP_XOR(nn.Module):
    def __init__(self, d_oculta=4):
        super().__init__()
        self.capa_oculta = nn.Linear(2, d_oculta)   # R² → R^d_oculta
        self.capa_salida = nn.Linear(d_oculta, 1)    # R^d_oculta → R
    
    def forward(self, x):
        h = torch.sigmoid(self.capa_oculta(x))   # representación oculta
        return torch.sigmoid(self.capa_salida(h)).squeeze()
    
    def representacion_oculta(self, x):
        """Devuelve las activaciones de la capa oculta."""
        with torch.no_grad():
            return torch.sigmoid(self.capa_oculta(x))


# Entrenar
torch.manual_seed(42)
mlp_xor = MLP_XOR(d_oculta=4)
opt_mlp = torch.optim.Adam(mlp_xor.parameters(), lr=0.05)

losses_mlp = []
for _ in range(3000):
    out = mlp_xor(X_xor)
    loss = crit(out, y_xor)
    opt_mlp.zero_grad()
    loss.backward()
    opt_mlp.step()
    losses_mlp.append(loss.item())

with torch.no_grad():
    probs_mlp = mlp_xor(X_xor)
    preds_mlp = (probs_mlp >= 0.5).float()

print("MLP (1 capa oculta, 4 neuronas) en XOR:")
print(f"  Loss final: {losses_mlp[-1]:.6f}")
print()
print(f"  {'Punto':>10} | {'Verdad':>8} | {'Predicción':>10} | {'Prob':>8}")
print("  " + "-"*45)
for nombre, yv, yp, prob in zip(nombres, y_xor, preds_mlp, probs_mlp):
    ok = "✓" if yv == yp else "✗"
    print(f"  {nombre:>10} | {int(yv.item()):>8} | {int(yp.item()):>10} {ok}  | {prob.item():>7.4f}")

In [ ]:
# La magia: visualizar la transformación del espacio

H = mlp_xor.representacion_oculta(X_xor).numpy()  # (4 puntos, 4 features ocultos)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Cómo la MLP transforma el espacio para resolver XOR', 
             fontsize=13, fontweight='bold')

colores_xor = ['#1565C0', '#C62828', '#C62828', '#1565C0']
marcadores = ['o', '^', '^', 'o']
etiquetas_short = ['(0,0)\ny=0', '(0,1)\ny=1', '(1,0)\ny=1', '(1,1)\ny=0']

# ── Espacio original ──
for i, (punto, color, marcador, etiq) in enumerate(zip(X_xor, colores_xor, marcadores, etiquetas_short)):
    axes[0].scatter(punto[0].item(), punto[1].item(), c=color, s=200,
                    marker=marcador, zorder=5, edgecolors='black', linewidth=1.5)
    axes[0].annotate(etiq, (punto[0].item(), punto[1].item()),
                     textcoords="offset points", xytext=(8, 5), fontsize=9)
axes[0].set_title('Espacio original\n$(x_1, x_2)$', fontsize=11)
axes[0].set_xlim(-0.3, 1.3); axes[0].set_ylim(-0.3, 1.3)
axes[0].set_xlabel('$x_1$'); axes[0].set_ylabel('$x_2$')
axes[0].grid(True, alpha=0.3)
axes[0].text(0.5, -0.25, '¿Separable linealmente? NO', ha='center', 
             color='red', fontsize=10, transform=axes[0].transAxes)

# ── Espacio oculto (dims 0 y 1) ──
for i, (h, color, marcador, etiq) in enumerate(zip(H, colores_xor, marcadores, etiquetas_short)):
    axes[1].scatter(h[0], h[1], c=color, s=200,
                    marker=marcador, zorder=5, edgecolors='black', linewidth=1.5)
    axes[1].annotate(etiq, (h[0], h[1]),
                     textcoords="offset points", xytext=(8, 5), fontsize=9)
axes[1].set_title('Espacio oculto\n$(h_1, h_2)$ — primeras 2 neuronas', fontsize=11)
axes[1].set_xlabel('$h_1$'); axes[1].set_ylabel('$h_2$')
axes[1].grid(True, alpha=0.3)
axes[1].text(0.5, -0.25, '¿Separable linealmente? SÍ (con 4 neuronas)', ha='center',
             color='green', fontsize=10, transform=axes[1].transAxes)

# ── Comparación de losses ──
axes[2].plot(losses_rl_xor, color='#E53935', linewidth=2, label='Reg. Logística (1 neurona)')
axes[2].plot(losses_mlp, color='#1E88E5', linewidth=2, label='MLP (capa oculta de 4)')
axes[2].axhline(0.01, color='green', linestyle=':', alpha=0.7, label='Loss ≈ 0.01')
axes[2].set_title('Loss: Reg. Logística vs MLP', fontsize=11)
axes[2].set_xlabel('Paso')
axes[2].set_ylabel('BCE Loss')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("La capa oculta aprende una transformación no lineal del espacio.")
print("En ese nuevo espacio, los datos sí son linealmente separables.")
print()
print("En lenguaje: la capa oculta construye 'conceptos' que no estaban")
print("en los tokens originales. Eso es lo que hace cada capa del Transformer.")

### Teorema de Aproximación Universal

> Una red neuronal con **una sola capa oculta** de suficiente tamaño puede aproximar cualquier función continua $f: \mathbb{R}^n \to \mathbb{R}$ en un dominio compacto, con precisión arbitraria.
> *(Hornik, Stinchcombe & White, 1989)*

Esto garantiza que la MLP tiene el poder expresivo para aprender cualquier patrón. El problema no es la capacidad teórica, sino:

1. **Encontrar** esos pesos — problema de optimización en alta dimensión
2. **Generalizar** a datos no vistos — regularización, dropout
3. **Escalar** eficientemente — arquitectura, paralelización

Los Transformers resuelven los tres: ADAM + residual connections resuelven (1), dropout + LayerNorm ayudan con (2), self-attention paralela resuelve (3).

In [ ]:
# ── Visualizar el poder expresivo: fronteras no lineales ────────────────────
# Dataset con forma de 'media luna' — no linealmente separable

from sklearn.datasets import make_moons

np.random.seed(42)
X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=42)
X_m = torch.tensor(X_moon, dtype=torch.float32)
y_m = torch.tensor(y_moon, dtype=torch.float32)

def entrenar_y_graficar(d_oculta, n_capas, ax, titulo):
    """Entrenar una MLP y visualizar su frontera de decisión."""
    torch.manual_seed(1)
    capas = [nn.Linear(2, d_oculta), nn.ReLU()]
    for _ in range(n_capas - 1):
        capas += [nn.Linear(d_oculta, d_oculta), nn.ReLU()]
    capas += [nn.Linear(d_oculta, 1), nn.Sigmoid()]
    modelo = nn.Sequential(*capas)
    
    opt = torch.optim.Adam(modelo.parameters(), lr=0.01)
    crit_bce = nn.BCELoss()
    for _ in range(1000):
        out = modelo(X_m).squeeze()
        loss = crit_bce(out, y_m)
        opt.zero_grad(); loss.backward(); opt.step()
    
    # Graficar
    x1r = np.linspace(X_moon[:,0].min()-0.3, X_moon[:,0].max()+0.3, 150)
    x2r = np.linspace(X_moon[:,1].min()-0.3, X_moon[:,1].max()+0.3, 150)
    xx1, xx2 = np.meshgrid(x1r, x2r)
    grid = torch.tensor(np.c_[xx1.ravel(), xx2.ravel()], dtype=torch.float32)
    with torch.no_grad():
        Z = modelo(grid).squeeze().numpy().reshape(xx1.shape)
    
    ax.contourf(xx1, xx2, Z, levels=50, cmap='RdYlBu_r', alpha=0.4)
    ax.contour(xx1, xx2, Z, levels=[0.5], colors='black', linewidths=1.5)
    ax.scatter(X_moon[y_moon==0, 0], X_moon[y_moon==0, 1], 
               c='#1565C0', s=20, alpha=0.8)
    ax.scatter(X_moon[y_moon==1, 0], X_moon[y_moon==1, 1], 
               c='#C62828', s=20, alpha=0.8)
    with torch.no_grad():
        acc = ((modelo(X_m).squeeze() >= 0.5).float() == y_m).float().mean().item()
    ax.set_title(f'{titulo}\nAcc: {acc*100:.1f}%', fontsize=10, fontweight='bold')
    ax.axis('off')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Poder expresivo: más capas → fronteras más complejas', 
             fontsize=12, fontweight='bold')

# Regresión logística
torch.manual_seed(1)
m_rl = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())
opt_rl = torch.optim.Adam(m_rl.parameters(), lr=0.01)
crit_b = nn.BCELoss()
for _ in range(1000):
    out = m_rl(X_m).squeeze(); loss = crit_b(out, y_m)
    opt_rl.zero_grad(); loss.backward(); opt_rl.step()
x1r = np.linspace(X_moon[:,0].min()-0.3, X_moon[:,0].max()+0.3, 150)
x2r = np.linspace(X_moon[:,1].min()-0.3, X_moon[:,1].max()+0.3, 150)
xx1, xx2 = np.meshgrid(x1r, x2r)
grid = torch.tensor(np.c_[xx1.ravel(), xx2.ravel()], dtype=torch.float32)
with torch.no_grad():
    Z_rl = m_rl(grid).squeeze().numpy().reshape(xx1.shape)
    acc_rl = ((m_rl(X_m).squeeze() >= 0.5).float() == y_m).float().mean().item()
axes[0].contourf(xx1, xx2, Z_rl, levels=50, cmap='RdYlBu_r', alpha=0.4)
axes[0].contour(xx1, xx2, Z_rl, levels=[0.5], colors='black', linewidths=1.5)
axes[0].scatter(X_moon[y_moon==0, 0], X_moon[y_moon==0, 1], c='#1565C0', s=20, alpha=0.8)
axes[0].scatter(X_moon[y_moon==1, 0], X_moon[y_moon==1, 1], c='#C62828', s=20, alpha=0.8)
axes[0].set_title(f'Reg. Logística\nAcc: {acc_rl*100:.1f}%', fontsize=10, fontweight='bold')
axes[0].axis('off')

entrenar_y_graficar(8,  1, axes[1], 'MLP: 1 capa oculta (8)')
entrenar_y_graficar(16, 2, axes[2], 'MLP: 2 capas ocultas (16)')
entrenar_y_graficar(32, 4, axes[3], 'MLP: 4 capas ocultas (32)')

plt.tight_layout()
plt.show()

---
## Parte 4: El Mapa Completo — de Aquí al Transformer

Todo lo que vimos hoy aparece directamente en el Transformer. Este es el mapa:

In [ ]:
# Mapa visual: regresión logística → MLP → Transformer

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#FAFAFA')
fig.patch.set_facecolor('#FAFAFA')

def caja(ax, x, y, w, h, texto, color_bg, color_borde, fontsize=10):
    rect = mpatches.FancyBboxPatch((x - w/2, y - h/2), w, h,
                                     boxstyle="round,pad=0.1",
                                     facecolor=color_bg, edgecolor=color_borde,
                                     linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, texto, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', color='white' if color_bg.startswith('#1') or color_bg.startswith('#0') or color_bg.startswith('#C') else 'black',
            wrap=True)

def flecha(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#424242', lw=2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.1, my + 0.1, label, fontsize=8, color='#424242')

# Fila 1: Regresión Logística
ax.text(7, 6.5, 'Hoy aprendimos:', ha='center', fontsize=11, style='italic', color='#555')
caja(ax, 2.5, 5.5, 4, 1.2, 'Regresión Logística\nσ(w·x + b)', '#1565C0', '#0D47A1', 10)
caja(ax, 7, 5.5, 4, 1.2, 'Cross-Entropy Loss\nMLE → backprop', '#C62828', '#B71C1C', 10)
caja(ax, 11.5, 5.5, 4, 1.2, 'MLP (capas ocultas)\nfronteras no lineales', '#2E7D32', '#1B5E20', 10)

flecha(ax, 4.5, 5.5, 5, 5.5, '')
flecha(ax, 9, 5.5, 9.5, 5.5, '')

# Fila 2: Conexiones
ax.text(7, 4.6, '↓  Es lo mismo que:', ha='center', fontsize=11, style='italic', color='#555')

caja(ax, 2.5, 3.8, 4, 1.2, 'Una neurona\nσ(z), activación', '#1565C0', '#0D47A1', 10)
caja(ax, 7, 3.8, 4, 1.2, 'Gradiente\n∂L/∂w = X·(ŷ - y)', '#C62828', '#B71C1C', 10)
caja(ax, 11.5, 3.8, 4, 1.2, 'Capas apiladas\nrepresentaciones', '#2E7D32', '#1B5E20', 10)

for x in [2.5, 7, 11.5]:
    ax.annotate('', xy=(x, 4.2), xytext=(x, 4.9),
                arrowprops=dict(arrowstyle='->', color='#9E9E9E', lw=1.5))

# Fila 3: Transformer
ax.text(7, 3.1, '↓  Aparece en el Transformer como:', ha='center', fontsize=11, style='italic', color='#555')

caja(ax, 2.5, 2.2, 4, 1.2, 'Feed-Forward Network\nLinear → ReLU → Linear', '#4527A0', '#311B92', 10)
caja(ax, 7, 2.2, 4, 1.2, 'Entrenamiento LLM\nAdam + scheduler', '#00695C', '#004D40', 10)
caja(ax, 11.5, 2.2, 4, 1.2, 'N bloques × 6 capas\nrepresentaciones contextuales', '#E65100', '#BF360C', 10)

for x in [2.5, 7, 11.5]:
    ax.annotate('', xy=(x, 2.7), xytext=(x, 3.2),
                arrowprops=dict(arrowstyle='->', color='#9E9E9E', lw=1.5))

# Referencia al paper
ax.text(7, 1.3, '"Attention is All You Need" — Sección 3.3:', 
        ha='center', fontsize=10, color='#555')
ax.text(7, 0.85, 'FFN(x) = max(0, xW₁ + b₁)W₂ + b₂', 
        ha='center', fontsize=11, fontfamily='monospace',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF9C4', edgecolor='#F9A825'))

plt.title('Del modelo más simple al Transformer', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Extensión natural: de binario a multiclase ───────────────────────────────
# La regresión logística binaria → softmax multiclase
# Esa misma softmax aparece en la ecuación de atención

print("Regresión Logística Binaria → Softmax Multiclase")
print("=" * 50)
print()
print("Binaria (2 clases):")
print("  ŷ = σ(w·x + b) = 1/(1 + e^{-z})")
print()
print("Multiclase (K clases):")
print("  ŷ_k = softmax(z)_k = e^{z_k} / Σ_j e^{z_j}")
print()
print("Son la misma idea: normalizar scores a una distribución de probabilidad.")
print()

# Demostración: sigmoide como caso especial de softmax 2 clases
z = torch.tensor(1.5)  # score de la clase 1
prob_sigmoid = torch.sigmoid(z)
prob_softmax = F.softmax(torch.tensor([0.0, z.item()]), dim=0)[1]  # z_0=0, z_1=z

print(f"Para z = {z.item()}:")
print(f"  sigmoid(z)           = {prob_sigmoid.item():.6f}")
print(f"  softmax([0, z])[1]   = {prob_softmax.item():.6f}")
print()
print("Son idénticos. La sigmoide es un caso especial de softmax con 2 clases.")
print()
print("En el Transformer (Ec. 1 del paper):")
print("  Attention(Q,K,V) = softmax(QK^T / √d_k) · V")
print()
print("El softmax ahí convierte scores de relevancia en pesos de atención.")
print("Misma operación, nuevo contexto.")

In [ ]:
# Preview: cómo se ve el Feed-Forward Network del Transformer
# vs nuestra MLP de hoy

print("=" * 58)
print("MLP de hoy  vs  FFN del Transformer")
print("=" * 58)

# Nuestra MLP_XOR
print()
print("MLP_XOR (hoy):")
mlp_comparar = MLP_XOR(d_oculta=4)
for nombre, p in mlp_comparar.named_parameters():
    print(f"  {nombre:30s} {str(p.shape):>20s}")

print()
print("FFN del Transformer (notebook siguiente):")
# FFN del paper: d_model=512, d_ff=2048
ffn_transformer = nn.Sequential(
    nn.Linear(512, 2048),
    nn.ReLU(),
    nn.Linear(2048, 512)
)
nombres_capas = ['linear1.weight', 'linear1.bias', 'linear2.weight', 'linear2.bias']
for (nombre, p), alias in zip(ffn_transformer.named_parameters(), nombres_capas):
    print(f"  {alias:30s} {str(p.shape):>20s}")

print()
print("Estructura idéntica. Solo cambia la escala (2→512 → 4→2048→512).")
print()
print("El Transformer apila 6 de estos bloques y agrega self-attention.")
print("La self-attention es la única parte verdaderamente nueva.")
print("Todo lo demás lo conocen.")

---
## Resumen de la Sesión

| Lo que vimos hoy | Dónde aparece en el Transformer |
|:----------------|:--------------------------------|
| `σ(w·x + b)` — una neurona | Cualquier capa lineal + activación |
| Cross-Entropy Loss + MLE | Función de pérdida en pretraining |
| Gradiente `∂L/∂w = X·(ŷ-y)` | Backpropagation en toda la red |
| Capa oculta → frontera no lineal | Feed-Forward Network (Sección 3.3) |
| `softmax` como generalización | Ecuación de atención (Ec. 1) |
| Profundidad → más poder expresivo | 6 bloques en encoder y decoder |

### Próximo notebook: `sesion_puente_redes_transformers.ipynb`

Ahora que la base está clara, el notebook puente construye cada componente del Transformer desde cero:
- **Scaled Dot-Product Attention** (la parte nueva)
- **Multi-Head Attention** 
- **Positional Encoding**
- Un **TransformerBlock** completo que referencia secciones específicas del paper

---
**Tarea antes de la siguiente clase:** Releer la Sección 3 del paper. Identificar en cada ecuación qué componente de hoy corresponde.

In [ ]:
# ── Ejercicio final: regresión logística multiclase en texto real ────────────
# Un dataset mínimo de NLP usando Bag of Words

from collections import Counter

# Dataset de intenciones simples
corpus = [
    ("qué hora es por favor", "pregunta"),
    ("cuánto cuesta esto", "pregunta"),
    ("dónde está el baño", "pregunta"),
    ("me gusta mucho el servicio", "positivo"),
    ("excelente producto muy bueno", "positivo"),
    ("lo recomiendo ampliamente", "positivo"),
    ("esto es una terrible pérdida", "negativo"),
    ("no me gustó para nada", "negativo"),
    ("muy mal servicio terrible", "negativo"),
]

# Vocabulario y vectorización BoW
vocab = sorted(set(w for texto, _ in corpus for w in texto.split()))
w2i = {w: i for i, w in enumerate(vocab)}
clases = ['pregunta', 'positivo', 'negativo']
c2i = {c: i for i, c in enumerate(clases)}

def vectorizar(texto):
    v = torch.zeros(len(vocab))
    for palabra in texto.split():
        if palabra in w2i:
            v[w2i[palabra]] = 1.0
    return v

X_nlp = torch.stack([vectorizar(t) for t, _ in corpus])
y_nlp = torch.tensor([c2i[c] for _, c in corpus])

# Regresión logística multiclase (softmax)
modelo_nlp = nn.Sequential(
    nn.Linear(len(vocab), 3)  # directo: vocab → 3 clases
    # Nota: nn.CrossEntropyLoss ya incluye softmax internamente
)

opt_nlp = torch.optim.Adam(modelo_nlp.parameters(), lr=0.05)
crit_ce = nn.CrossEntropyLoss()

for epoch in range(500):
    logits = modelo_nlp(X_nlp)
    loss = crit_ce(logits, y_nlp)
    opt_nlp.zero_grad(); loss.backward(); opt_nlp.step()

with torch.no_grad():
    logits_final = modelo_nlp(X_nlp)
    probs_final = F.softmax(logits_final, dim=-1)
    preds_final = logits_final.argmax(dim=-1)

print("Regresión Logística Multiclase — Clasificación de Intenciones")
print(f"Vocabulario: {len(vocab)} palabras")
print(f"Loss final: {loss.item():.4f}")
print()
print(f"{'Texto':>35} | {'Verdad':>9} | {'Pred':>9} | {'Probs':<30}")
print("-" * 90)
for (texto, clase), pred, prob in zip(corpus, preds_final, probs_final):
    ok = "✓" if c2i[clase] == pred.item() else "✗"
    p_str = f"[{prob[0]:.2f}, {prob[1]:.2f}, {prob[2]:.2f}]"
    print(f"{texto[:35]:>35} | {clase:>9} | {clases[pred.item()]:>8}{ok} | {p_str}")

print()
print("Este es exactamente el mismo modelo que BERT usa para clasificación:")
print("embedding del token [CLS] → nn.Linear → softmax → clase")
print("La única diferencia es que el embedding de BERT es riquísimo (768 dims)")
print("y se obtuvo con el Transformer que estudiaremos.")